In [9]:
print(df.columns)


Index(['LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default'], dtype='object')


In [10]:
file_path = r"D:\Desktop\CreditPathAI\data\Loan_default.csv"

# Load using comma separator and ensure proper header
df = pd.read_csv(file_path, sep=",")  # Use ',' instead of '\t'

# Strip any extra spaces in column names
df.columns = df.columns.str.strip()

print("Columns loaded:", df.columns)
print("First 5 rows:")
print(df.head())


Columns loaded: Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='object')
First 5 rows:
       LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0               4         15.23        36      0.44   Bachelor's   
1               1          4.81        60      0.68     Master's   
2               3         21

In [12]:
# --- Step 1: Import Libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from tabulate import tabulate
import warnings
import os

warnings.filterwarnings("ignore")

# --- Step 2: Load your dataset ---
file_path = r"D:\Desktop\CreditPathAI\data\Loan_default.csv"  # Update to your path
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

df = pd.read_csv(file_path, sep=",")  # Change sep="\t" if your file is tab-separated
df.columns = df.columns.str.strip()   # Remove extra spaces from column names

# --- Step 3: Target Column ---
# If your dataset already has a column like 'Default' use it
# Otherwise, create target based on status column if available
if "Default" in df.columns:
    y = df["Default"]
else:
    target_col = "loanStatus"  # Replace with your status column
    df["Default"] = df[target_col].apply(lambda x: 1 if x in ["Default", "Charged Off", "Late"] else 0)
    y = df["Default"]

# --- Step 4: Features & Target ---
# Drop irrelevant columns for modeling
drop_cols = ["LoanID", "Default"]
if "loanId" in df.columns: drop_cols.append("loanId")
if "memberId" in df.columns: drop_cols.append("memberId")
if "date" in df.columns: drop_cols.append("date")

X = df.drop(columns=drop_cols, errors='ignore')

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Impute missing values
X = SimpleImputer(strategy="median").fit_transform(X)

# Train/Validation/Test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# --- Step 5: Function to evaluate model ---
def evaluate_model(model, model_name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
    
    return [
        model_name,
        round(accuracy_score(y_train, y_train_pred), 4),
        round(accuracy_score(y_test, y_test_pred), 4),
        round(recall_score(y_test, y_test_pred), 4),  # Sensitivity
        round(tn / (tn + fp), 4),                    # Specificity
        round(precision_score(y_test, y_test_pred), 4),
        round(f1_score(y_test, y_test_pred), 4)
    ]

# --- Step 6: Run all models ---
models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, random_state=42)),
    ("KNN", KNeighborsClassifier(n_neighbors=5)),
    ("Decision Tree", DecisionTreeClassifier(random_state=42)),
    ("Random Forest", RandomForestClassifier(n_estimators=100, random_state=42)),
    ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ("XGBoost", XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)),
    ("Linear SVM", LinearSVC(max_iter=5000, random_state=42))
]

results = []
for name, model in models:
    results.append(evaluate_model(model, name))

# --- Step 7: Display results in a single table ---
headers = ["Model", "Train Acc", "Test Acc", "Sensitivity", "Specificity", "Precision", "F1-Score"]
print(tabulate(results, headers=headers, tablefmt="grid"))


Columns in dataset: Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='object')
First 5 rows:
        LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0               4         15.23        36      0.44   Bachelor's   
1               1          4.81        60      0.68     Master's   
2               3      

In [16]:
# --- Step 1: Import Libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import LinearSVC
from tabulate import tabulate
import warnings
import os

warnings.filterwarnings("ignore")

# --- Step 2: Load your dataset ---
file_path = r"D:\Desktop\CreditPathAI\data\Loan_default.csv"  # Update to your path
if not os.path.exists(file_path):
    raise FileNotFoundError(f"File not found: {file_path}")

df = pd.read_csv(file_path, sep=",")  # Change sep="\t" if your file is tab-separated
df.columns = df.columns.str.strip()   # Remove extra spaces from column names

# --- Step 3: Target Column ---
# If your dataset already has a column like 'Default' use it
# Otherwise, create target based on status column if available
if "Default" in df.columns:
    y = df["Default"]
else:
    target_col = "loanStatus"  # Replace with your status column
    df["Default"] = df[target_col].apply(lambda x: 1 if x in ["Default", "Charged Off", "Late"] else 0)
    y = df["Default"]

# --- Step 4: Features & Target ---
# Drop irrelevant columns for modeling
drop_cols = ["LoanID", "Default"]
if "loanId" in df.columns: drop_cols.append("loanId")
if "memberId" in df.columns: drop_cols.append("memberId")
if "date" in df.columns: drop_cols.append("date")

X = df.drop(columns=drop_cols, errors='ignore')

# One-hot encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Impute missing values
X = SimpleImputer(strategy="median").fit_transform(X)

# Train/Validation/Test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# --- Step 5: Function to evaluate model ---
def evaluate_model(model, model_name):
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
    
    return [
        model_name,
        round(accuracy_score(y_train, y_train_pred), 4),
        round(accuracy_score(y_test, y_test_pred), 4),
        round(recall_score(y_test, y_test_pred), 4),  # Sensitivity
        round(tn / (tn + fp), 4),                    # Specificity
        round(precision_score(y_test, y_test_pred), 4),
        round(f1_score(y_test, y_test_pred), 4)
    ]

# --- Step 6: Run all models ---
models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("KNN", KNeighborsClassifier(n_neighbors=5)),
    ("Decision Tree", DecisionTreeClassifier()),
    ("Random Forest", RandomForestClassifier(n_estimators=100)),
    ("Gradient Boosting", GradientBoostingClassifier(n_estimators=100)),
    ("XGBoost", XGBClassifier(eval_metric='logloss', use_label_encoder=False)),
    ("Linear SVM", LinearSVC(max_iter=5000))
]

results = []
for name, model in models:
    results.append(evaluate_model(model, name))

# --- Step 7: Display results in a single table ---
headers = ["Model", "Train Acc", "Test Acc", "Sensitivity", "Specificity", "Precision", "F1-Score"]
print(tabulate(results, headers=headers, tablefmt="grid"))


+---------------------+-------------+------------+---------------+---------------+-------------+------------+
| Model               |   Train Acc |   Test Acc |   Sensitivity |   Specificity |   Precision |   F1-Score |
+=====================+=============+============+===============+===============+=============+============+
| Logistic Regression |      0.885  |     0.8854 |        0.0344 |        0.9972 |      0.6145 |     0.0651 |
+---------------------+-------------+------------+---------------+---------------+-------------+------------+
| KNN                 |      0.8943 |     0.875  |        0.0528 |        0.983  |      0.2898 |     0.0894 |
+---------------------+-------------+------------+---------------+---------------+-------------+------------+
| Decision Tree       |      1      |     0.8024 |        0.2304 |        0.8775 |      0.1982 |     0.2131 |
+---------------------+-------------+------------+---------------+---------------+-------------+------------+
| Random F